In [ ]:
import json
import numpy as np
from scipy.stats import mannwhitneyu

def vargha_delaney_A12(a, b):
    """
    U: Mann-Whitney U statistic

    A12 > 0.5  -> a is better
    A12 = 0.5  -> no difference
    A12 < 0.5  -> b is better
    """
    common_bugs = sorted(set(a) & set(b))
    a_res = [a[cb] for cb in common_bugs]
    b_res = [b[cb] for cb in common_bugs]

    U, p_value = mannwhitneyu(a_res, b_res, alternative="two-sided")

    A12_U, _ = mannwhitneyu(b_res, a_res, alternative="two-sided")

    A12 = A12_U / (len(a_res) * len(b_res))

    return round(U, 3), round(p_value, 3), round(A12, 3)

def extract_best_ranks(data):
    bug_ranks = {}

    for bug, methods in data["buggy_methods"].items():
        ranks = []

        for m in methods.values():
            ranks.append(m["autofl_rank"])

        if ranks:
            bug_ranks[bug] = min(ranks)

    return bug_ranks

def get_ranks(exp_name, gpt_version, repetition):
    score_path_json = f"../combined_fl_results/{exp_name}/{gpt_version}_R{repetition}_full_light.json"
    with open(score_path_json, "r") as f:
        data = json.load(f)
        return extract_best_ranks(data)

def compare_with_baselines(target_exp, GPT_VERSION = "gpt-3.5-turbo-0125"):
    lines = []
    for rep in [1, 5]:
        # baselines
        RFL_CR = get_ranks("current", GPT_VERSION, rep)
        AFL_ONLYFT = get_ranks("autofl_onlyft", GPT_VERSION, rep)
        AFL = get_ranks("autofl", GPT_VERSION, rep)

        # target exp ranks
        target_ranks = get_ranks(target_exp, GPT_VERSION, rep)

        RFL_CR_U, RFL_CR_p_value, RFL_CR_A12 = vargha_delaney_A12(target_ranks, RFL_CR)
        AFL_ONLYFT_U, AFL_ONLYFT_p_value, AFL_ONLYFT_A12 = vargha_delaney_A12(target_ranks, AFL_ONLYFT)
        AFL_U, AFL_p_value, AFL_A12 = vargha_delaney_A12(target_ranks, AFL)
        lines.append(f"& {RFL_CR_A12} & {AFL_ONLYFT_A12} & {AFL_A12}")
        lines.append(f"& {RFL_CR_p_value} & {AFL_ONLYFT_p_value} & {AFL_p_value}")
    
    print("\n".join(lines))


In [32]:
print("& \\namedays{30}")
compare_with_baselines("reportfl")
print("& \\namedays{60}")
compare_with_baselines("reportfl_60days")
print("& \\namedays{90}")
compare_with_baselines("reportfl_90days")

& \namedays{30}
& 0.492 & 0.502 & 0.362
& 42460.5 & 41626.0 & 52948.5
& 0.494 & 0.535 & 0.452
& 42223.5 & 38852.0 & 45431.0
& \namedays{60}
& 0.502 & 0.511 & 0.375
& 41601.0 & 40806.5 & 51860.5
& 0.483 & 0.522 & 0.44
& 43203.0 & 39956.0 & 46461.5
& \namedays{90}
& 0.522 & 0.533 & 0.403
& 39949.5 & 39021.0 & 49538.0
& 0.493 & 0.532 & 0.451
& 42338.0 & 39072.5 & 45568.5


In [33]:
print("& \\namedays{30}")
compare_with_baselines("reportfl", GPT_VERSION="gpt-4.1-mini-2025-04-14")
print("& \\namedays{60}")
compare_with_baselines("reportfl_60days", GPT_VERSION="gpt-4.1-mini-2025-04-14")
print("& \\namedays{90}")
compare_with_baselines("reportfl_90days", GPT_VERSION="gpt-4.1-mini-2025-04-14")

& \namedays{30}
& 0.503 & 0.572 & 0.546
& 41487.5 & 35783.5 & 37697.5
& 0.505 & 0.563 & 0.569
& 41365.0 & 36496.0 & 35713.5
& \namedays{60}
& 0.512 & 0.579 & 0.554
& 40783.5 & 35185.5 & 36981.0
& 0.514 & 0.571 & 0.577
& 40594.5 & 35839.5 & 35082.5
& \namedays{90}
& 0.496 & 0.562 & 0.534
& 42122.0 & 36603.0 & 38617.0
& 0.512 & 0.57 & 0.577
& 40734.0 & 35896.0 & 35044.5
